# Model calibration

A classifier can be **accurate but miscalibrated**: when it says "90% sure," is
it actually right 90% of the time? Calibration asks whether the predicted
*probabilities* are trustworthy — which matters whenever you act on the number,
not just the label (risk scores, thresholds, expected-value decisions).

This builds directly on [logistic regression](../02-regression/logistic-regression.ipynb)
(its cross-entropy training *should* give calibrated probabilities) and the
[evaluation chapter](../01d-evaluation/cross-validation.ipynb).

In [ ]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
:dep plotters-statistical = { version = "0.2.0" }

fn sigmoid(z: f64) -> f64 { 1.0 / (1.0 + (-z).exp()) }

// True world: P(y=1 | x) = sigmoid(0.8*x). Labels are SAMPLED from that true
// probability (Bernoulli), so observed frequencies really follow sigmoid(0.8*x).
// The 'model' is OVERCONFIDENT: it reports sigmoid(2*x) — far too steep.
let n = 800usize;
let x: Vec<f64> = (0..n).map(|i| (i as f64 / n as f64) * 8.0 - 4.0).collect();
let y: Vec<f64> = (0..n).map(|i| {
    let u = ((i.wrapping_mul(1103515245).wrapping_add(12345) >> 16) & 0x7fff) as f64 / 32768.0;  // pseudo-uniform in [0,1)
    if u < sigmoid(0.8 * x[i]) { 1.0 } else { 0.0 }
}).collect();
let model_prob: Vec<f64> = (0..n).map(|i| sigmoid(2.0 * x[i]).clamp(1e-4, 1.0 - 1e-4)).collect();

// The reliability diagram itself now comes from `plotters-statistical`'s
// CalibrationCurve (below); we keep the Brier score by hand as our summary number.
fn brier(p: &[f64], y: &[f64]) -> f64 { p.iter().zip(y).map(|(a, b)| (a - b).powi(2)).sum::<f64>() / p.len() as f64 }
println!("{} samples; labels sampled from the true prob, overconfident model built", n);

## The reliability diagram

Bin the predictions, and for each bin plot **mean predicted probability** (x) vs.
**observed frequency of y=1** (y). Perfect calibration lies on the diagonal.
`plotters-statistical`'s `CalibrationCurve::from_scores` does the binning and draws
the curve (predictions + boolean labels + bin count). Our overconfident model bows
*away* from the diagonal — high predictions over-promise, low predictions
under-promise:

In [ ]:
{
    use plotters::prelude::*;
    use plotters_statistical::CalibrationCurve;
    let labels: Vec<bool> = y.iter().map(|&v| v > 0.5).collect();
    println!("Brier score (lower = better): {:.4}", brier(&model_prob, &y));
    evcxr_figure((420, 400), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("Reliability: overconfident model", ("sans-serif", 14)).margin(10).x_label_area_size(30).y_label_area_size(40).build_cartesian_2d(0f64..1f64, 0f64..1f64)?;
        c.configure_mesh().x_desc("mean predicted prob").y_desc("observed frequency").draw()?;
        // from_scores(scores, boolean labels, n_bins); .diagonal(true) draws the
        // perfect-calibration reference line.
        c.draw_series(std::iter::once(
            CalibrationCurve::from_scores(&model_prob, &labels, 10)?.color(RED).diagonal(true)))?;
        Ok(())
    })
}

## Fixing it: Platt scaling

**Platt scaling** fits a small logistic regression on the model's *scores* to
remap them: `p_calibrated = sigmoid(A·logit(p) + B)`. We fit `A, B` by gradient
descent (the [logistic GD](../02-regression/logistic-regression.ipynb) again) —
`A < 1` will pull the overconfident logits back toward the middle:

In [ ]:
let calibrated: Vec<f64> = {
    let logit = |p: f64| (p / (1.0 - p)).ln();
    let (mut a, mut b) = (1.0, 0.0);
    for _ in 0..3000 {
        let (mut ga, mut gb) = (0.0, 0.0);
        for i in 0..n { let z = logit(model_prob[i]); let e = sigmoid(a * z + b) - y[i]; ga += e * z; gb += e; }
        a -= 0.1 * ga / n as f64; b -= 0.1 * gb / n as f64;
    }
    println!("Platt parameters: A = {:.3} (<1 => was overconfident), B = {:.3}", a, b);
    (0..n).map(|i| sigmoid(a * logit(model_prob[i]) + b)).collect()
};
println!("Brier before = {:.4}, after = {:.4}", brier(&model_prob, &y), brier(&calibrated, &y));

The Brier score drops (better) and `A < 1` confirms the model was overconfident.
The calibrated reliability curve now hugs the diagonal:

In [ ]:
{
    use plotters::prelude::*;
    use plotters_statistical::CalibrationCurve;
    let labels: Vec<bool> = y.iter().map(|&v| v > 0.5).collect();
    evcxr_figure((420, 400), |root| {
        root.fill(&WHITE)?;
        let mut c = ChartBuilder::on(&root).caption("Before (red) vs after Platt (green)", ("sans-serif", 14)).margin(10).x_label_area_size(30).y_label_area_size(40).build_cartesian_2d(0f64..1f64, 0f64..1f64)?;
        c.configure_mesh().x_desc("mean predicted prob").y_desc("observed frequency").draw()?;
        c.draw_series(std::iter::once(
            CalibrationCurve::from_scores(&model_prob, &labels, 10)?.color(RED).diagonal(true)))?;
        c.draw_series(std::iter::once(
            CalibrationCurve::from_scores(&calibrated, &labels, 10)?.color(GREEN)))?;
        Ok(())
    })
}

## In practice

- **Fit the calibrator on held-out data**, never the training set (it would just
  memorise) — use the [evaluation chapter's](../01d-evaluation/cross-validation.ipynb)
  split, exactly as you would for any other model.
- **Platt scaling** (logistic, 2 parameters) is simple and data-efficient;
  **isotonic regression** is more flexible but needs more data and has no
  maintained Rust crate — a hand-rolled pool-adjacent-violators fit or a coarse
  binning is the fallback.
- Well-trained logistic regression is often already calibrated; tree ensembles
  and SVMs frequently are **not**, which is exactly when this chapter earns its
  keep.

Report calibration (a reliability diagram + Brier score) alongside the
[accuracy/AUC metrics](../01d-evaluation/metrics-deep-dive.ipynb) — a model can be
accurate and still lie about its confidence.